In [3]:
%load_ext autoreload
%autoreload 2
%cd /home/albin/egna_proj/block_puzzle_rl/

/home/albin/egna_proj/block_puzzle_rl


/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
# ⬇️ SB3 + Gym imports
import numpy as np
import torch
import time
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from gymnasium import spaces
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np

# ⬇️ Your imports
from game.block_puzzle_env import BlockPuzzleEnv
from agent.utils import encode_state


In [5]:

# ⬇️ Custom wrapper to convert dict obs → flat array
class FlattenedBlockEnv(gym.Env):
    def __init__(self, width=12, height=12, num_blocks=3):
        super().__init__()
        self.raw_env = BlockPuzzleEnv(width, height, num_blocks)
        self.action_space = self.raw_env.action_space
        dummy_obs, _ = self.raw_env.reset()
        sample_obs = encode_state(dummy_obs)
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=sample_obs.shape, dtype=np.float32
        )
    
    def reset(self, seed=None, options=None):
        obs_dict, _ = self.raw_env.reset()
        flat_obs = encode_state(obs_dict)
        return flat_obs.astype(np.float32), {}

    def step(self, action):
        # Handle batched action from DummyVecEnv (e.g., [0, 9, 10])
        if isinstance(action, (list, np.ndarray)) and len(action) == 3:
            a0, a1, a2 = map(int, action)
        else:
            # Flat index → unravel into (block_index, row, col)
            a0, a1, a2 = np.unravel_index(action, self.action_space.nvec)

        raw_action = (a0, a1, a2)
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(raw_action)
        flat_obs = encode_state(obs_dict)
        return flat_obs.astype(np.float32), reward, terminated, truncated, info


    def render(self):
        return self.raw_env.render()

    def close(self):
        return self.raw_env.close()


class DiscreteActionWrapper(gym.Env):
    def __init__(self, raw_env):
        super().__init__()
        self.raw_env = raw_env
        self.original_action_space = raw_env.action_space  # Should be MultiDiscrete
        self.obs_space = self.raw_env.observation_space

        # Flatten MultiDiscrete([a, b, c]) → Discrete(a * b * c)
        self.action_space = spaces.Discrete(np.prod(self.original_action_space.nvec))
        self.observation_space = spaces.Box(low=0, high=1, shape=(encode_state(self.raw_env.reset()[0]).shape[0],), dtype=np.float32)

    def reset(self, **kwargs):
        obs_dict, _ = self.raw_env.reset(**kwargs)
        return encode_state(obs_dict).astype(np.float32), {}

    def step(self, flat_action):
        a0, a1, a2 = np.unravel_index(flat_action, self.original_action_space.nvec)
        action = (int(a0), int(a1), int(a2))
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(action)
        flat_obs = encode_state(obs_dict)
        return flat_obs.astype(np.float32), reward, terminated, truncated, info

    def render(self, **kwargs):
        return self.raw_env.render(**kwargs)



In [6]:
class CustomMetricsCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)

    def _on_step(self):
        infos = self.locals["infos"]
        for info in infos:
            for key, value in info.items():
                if key in ['TimeLimit.truncated', 'terminal_observation']:
                    continue
                if isinstance(value, np.ndarray):
                    self.logger.record(f"custom/{key}", value.mean())
                else:
                    self.logger.record(f"custom/{key}", value)
        return True

In [7]:
raw_env = BlockPuzzleEnv(width=12, height=12, num_blocks=3)
wrapped_env = DiscreteActionWrapper(raw_env)
monitored_env = Monitor(wrapped_env)

check_env(wrapped_env, warn=True)  # ✅ Now this should pass
vec_env = DummyVecEnv([lambda: monitored_env])

model = DQN(
    policy="MlpPolicy",
    env=vec_env,
    learning_rate=1e-3,
    buffer_size=50000,
    learning_starts=1000,
    batch_size=512,
    gamma=0.99,
    train_freq=4,
    target_update_interval=100,
    exploration_fraction=0.3,
    exploration_final_eps=0.05,
    verbose=1,
    tensorboard_log="./sb3_logs/"
)

callback = CustomMetricsCallback()
# model.learn(total_timesteps=200000, callback=callback)
# model.save("sb3_block_dqn")
model = DQN.load("sb3_block_dqn", env=vec_env)

Using cuda device


In [16]:
obs, _ = wrapped_env.reset()
flat_action, mask = model.predict(obs, deterministic=False)
np.unravel_index(flat_action, wrapped_env.original_action_space.nvec)

(np.int64(1), np.int64(0), np.int64(4))

In [10]:
obs, _ = wrapped_env.reset()
done = False
total_reward = 0

while not done:
    action, _ = model.predict(obs, deterministic=False)
    obs, reward, terminated, truncated, info = wrapped_env.step(action)
    total_reward += reward
    done = terminated or truncated
    wrapped_env.render()
    time.sleep(0.02)  
    print()

print(f"Total reward (greedy run): {total_reward}")

□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ ■ □ □ □
□ □ □ □ □ □ □ □ ■ ■ □ □
□ □ □ □ □ □ □ □ □ ■ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ ■ □ □ □
□ □ □ □ □ □ □ ■ ■ □ □ □
□ □ □ □ □ □ □ ■ ■ □ □ □
□ □ □ □ □ □ □ □ ■ ■ □ □
□ □ □ □ □ □ □ □ □ ■ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ ■ □ □ □
□ □ □ □ □ □ □ ■ ■ □ □ □
□ □ □ □ □ □ □ ■ ■ □ □ □
□ □ □ □ □ □ □ □ ■ ■ □ □
□ □ □ □ □ □ □ □ □ ■ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ ■ □ □ □
□ □ □ □ □ □ □ ■ ■ □ □ □
□ □ □ □ □ □ □ ■ ■ □ □ □
□ □ □ □ □ □ □

KeyboardInterrupt: 